# Interactive Notebook 04 - PMSM basic control:

This interactive Jupyter notebook introduces basic control concepts for permanent magnet synchronous motors.

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.io import loadmat
from pathlib import Path
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})

import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.)

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---

**Throught the whole notebook, we will assume that the rotation speed $\omega$ can be set arbitrarily from an external load machine.**

### Control of the nonlinear PMSM:

We will reuse the implementation from the simulation exercise as our plant to be controlled (code in `pmsm.py`).

Now we want to control the system to stay at a specific current operating point $\boldsymbol{i}^*_{\mathrm{s, dq}}$.

In [ ]:
# starting with this notebook, all auxiliary simulation code for the PMSM will be placed in 
# `CTPD/interactive_notebooks/utils/`
import sys

from path_helper import get_folder_path
sys.path.insert(1, str(get_folder_path()))

In [ ]:
from utils.pmsm import PMSM, lut_interpolate
from utils.motor_params import MotorVariant
from utils.visualization import visualize_trajectories, visualize_trajectories_with_reference
from utils.signals import aprbs

In [ ]:
pmsm = PMSM(
    saturated=True,
    motor_variant=MotorVariant.BRUSA,
    T_s=1e-4,
)
pmsm

In [ ]:
obs, state = pmsm.reset()

print(obs)
print(state)

In [ ]:
# from motor_params import NonlinearParams, generate_luts, lut_interpolate
# from pmsm import nonlinear_ode, nonlinear_euler_step

In [ ]:
# lut_raw = loadmat(Path("LUT_BRUSA_jax_grad.mat"))
# lut_grids, lut_values = generate_luts(lut_raw)

# nonlinear_motor_params = NonlinearParams(
#     p=3,
#     R_s=jnp.array(17.932e-3),
#     lut_grids=lut_grids,
#     lut_values=lut_values,
#     i_lim=250,
#     u_dc=400,
# )

#### Dummy controller:

First without control delay.

In [ ]:
def dummy_controller(i_dq):
    u_dq = jnp.array([0.0, 0.0])
    return u_dq

In [ ]:
state

In [ ]:
# simulate controller plant interaction:
sequence_length = 10_000

obs, state = pmsm.reset() # puts the state back to the equilibrium point, zero-speed
obs, state = pmsm.set_speed(n=500, state=state)

i_dq = obs[:2]

i_dq_sequence = [i_dq]
u_dq_sequence = []

for i in range(sequence_length):
    u_dq = dummy_controller(i_dq)

    obs, state = pmsm.step(state, u_dq)
    i_dq_next = obs[:2]

    i_dq = i_dq_next
    
    i_dq_sequence.append(i_dq)
    u_dq_sequence.append(u_dq)

i_dq_sequence = jnp.array(i_dq_sequence)
u_dq_sequence = jnp.array(u_dq_sequence)

visualize_trajectories(
    i_dq_sequences=[i_dq_sequence],
    u_dq_sequences=[u_dq_sequence], 
    T_s_list=[pmsm.T_s],
    omegas=[state.physical_state.omega_el],
    ode=None,
    params=pmsm.env_properties.static_params,
)

#### FOC-PI:

including control delay:

In [ ]:
class FOCController:

    def __init__(self, static_params, lut_grids, lut_values, a, T_s):
        self.T_s = T_s
        self.a = a
        self.static_params = static_params
        self.lut_grids = lut_grids
        self.lut_values = lut_values

    def integrate(self, integrated, e):
        integrated = (integrated + e * self.T_s)
        return integrated

    def reset(self):
        return jnp.zeros((2,))
    
    @eqx.filter_jit
    def __call__(self, i_dq, i_dq_ref, omega, integrated):
        e = i_dq_ref - i_dq

        # get parameters for operating point
        p_d = {q: lut_interpolate(*self.lut_grids[q], self.lut_values[q], *i_dq) for q in self.lut_grids}
        L_dq = jnp.column_stack([p_d[q] for q in ["L_dd", "L_qq"]])
        psi_dq = jnp.column_stack([p_d[q] for q in ["Psi_d", "Psi_q"]])

        # compute gains
        p_gain = 1 / (self.a * 1.5 * self.T_s) * L_dq
        p_gain_help = 1 / (self.a * 1.5 * self.T_s) * L_dq
        i_gain = 0.3 * p_gain_help / ((self.a) ** 2 * 1.5 * self.T_s)

        q = jnp.array([0, -1, 1, 0]).reshape(2, 2)

        u_s_0 = omega * jnp.einsum("ij,bj->bi", q, psi_dq)

        integrated = self.integrate(integrated, e)
        u_dq = p_gain * e + i_gain * integrated + u_s_0
        return jnp.squeeze(u_dq), integrated

In [ ]:
# simulate controller plant interaction:
sequence_length = 1_000

obs, state = pmsm.reset() # puts the state back to the equilibrium point, zero-speed
obs, state = pmsm.set_speed(n=500, state=state)

controller = FOCController(
    static_params=pmsm.env_properties.static_params,
    lut_grids=pmsm.LUT_grids,
    lut_values=pmsm.LUT_values,
    a=6,
    T_s=pmsm.T_s
) 
integrated = controller.reset()

i_dq_ref_sequence = aprbs(sequence_length, 2, t_min=50, t_max=300, key=jax.random.key(0)).T[0] * 200

i_dq = obs[:2]
u_dq = jnp.array([0.0, 0.0])

i_dq_sequence = [i_dq]
u_dq_sequence = [u_dq]

for i_dq_ref in i_dq_ref_sequence:
    u_dq_next, integrated = controller(i_dq, i_dq_ref, state.physical_state.omega_el, integrated)

    obs, state = pmsm.step(state, u_dq)
    i_dq_next = obs[:2]

    i_dq = i_dq_next
    u_dq = u_dq_next
    
    i_dq_sequence.append(i_dq)
    u_dq_sequence.append(u_dq)

i_dq_sequence = jnp.array(i_dq_sequence)
u_dq_sequence = jnp.array(u_dq_sequence)

visualize_trajectories_with_reference(
    i_dq_sequences=[i_dq_sequence],
    u_dq_sequences=[u_dq_sequence], 
    i_dq_ref_sequence=i_dq_ref_sequence,
    T_s_list=[pmsm.T_s],
    omegas=[state.physical_state.omega_el],
    ode=None,
    params=pmsm.env_properties.static_params,
)

#### New PI:

#### Anti windup

In [ ]:
raise NotImplementedError()

#### Anti wind-up: